[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seap-udea/fargopy/blob/main/examples/legacy/fargopy-tutorial-plotly.ipynb)

<p align="left"><img src="https://github.com/seap-udea/fargopy/raw/refactor/docs/fargopy_logo.webp" width="300" /></p>

# Tutorial: using `plotly`

In [1]:
try:
    from google.colab import drive
    %pip install -Uq git+https://github.com/seap-udea/fargopy
except ImportError:
    print("Not running in Colab, skipping installation")
    %load_ext autoreload
    %autoreload 2
!mkdir -p ./gallery/


Not running in Colab, skipping installation


### What's in this notebook

In this notebook we illustrate **how to use `plotly` capabilities to plot simulation results** in `FARGOpy`.

### Before starting

If you are in `Google Colab`, install the latest version of the package:

For this tutorial you will need the following external modules and tools:

In [3]:
import fargopy as fp
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go


### Let's `FARGOpy`

Let's download a precomputed simulation:

In [4]:
fp.Simulation.download_precomputed('p3disoj')

Precomputed output directory '/tmp/p3disoj' already exist


'/tmp/p3disoj'

Load the simulation and read the density field:

In [5]:
# Connect to the simulation results
sim = fp.Simulation(output_dir='/tmp/p3disoj')

# Get fields and slice them
snap = 10
gasdens = sim.load_field('gasdens',type='scalar',snapshot=snap,interpolate=True)

FARGO3D directory '/Users/jzuluaga/fargo3d/' does not exist.
Now you are connected with output directory '/tmp/p3disoj'
Found a variables.par file in '/tmp/p3disoj', loading properties
Loading variables
85 variables loaded
Simulation in 3 dimensions
Loading domain in spherical coordinates:
	Variable phi: 128 [[0, np.float64(-3.117048960983623)], [-1, np.float64(3.117048960983623)]]
	Variable r: 64 [[0, np.float64(0.5078125)], [-1, np.float64(1.4921875)]]
	Variable theta: 32 [[0, np.float64(1.4231400767948967)], [-1, np.float64(1.5684525767948965)]]
Number of snapshots in output directory: 11
Planets found in summary.dat:
  Name: Jupiter, Initial pos: [1.0, 0.001, 0.0], Mass: 0.001


We want to create a 3 dimensional representation of the field. First let's slice the filed on a plane of constant $\varphi$:

In [6]:
slice = "phi=0"
gasdens_slice,mesh = gasdens.meshslice(slice=slice)

AttributeError: 'FieldInterpolator' object has no attribute 'meshslice'

Now create a 3D scatter plot of the field:

In [6]:
xmin = mesh.x.min()
xmax = mesh.x.max()
range = xmax - xmin
ymin = mesh.y.min() - range/2
ymax = mesh.y.max() + range/2
zmin = mesh.z.min()
zmax = mesh.z.max()


In [7]:
from IPython.display import display, HTML
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=mesh.x.flatten(),
        y=mesh.y.flatten(),
        z=mesh.z.flatten(),
        mode='markers',
        marker=dict(
            size=4,
            color=np.log10(gasdens_slice).flatten(),
            colorscale='thermal',
            opacity=0.5
        )
    )
)

fig.update_layout(
    autosize=False,
    width=600,
    height=600,
    scene=dict(
        aspectmode='cube',
        xaxis=dict(range=[xmin, xmax]),
        yaxis=dict(range=[ymin, ymax]),
        zaxis=dict(range=[zmin, zmax]),
        #xaxis_visible=False,
        #yaxis_visible=False,
        #zaxis_visible=False,
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
)

# Show the plot
fig.show()

Let's do it at different angles:

In [8]:
from IPython.display import display, HTML
fig = go.Figure()

dphi = 15
for phi in np.arange(0,90+dphi,dphi):

    slice = f"phi={phi} deg"
    gasdens_slice,mesh = gasdens.meshslice(slice=slice)

    fig.add_trace(
        go.Scatter3d(
            x=mesh.x.flatten(),
            y=mesh.y.flatten(),
            z=mesh.z.flatten(),
            mode='markers',
            name='phi = '+str(phi)+' deg',
            marker=dict(
                size=5,
                color=np.log10(gasdens_slice).flatten(),
                colorscale='thermal',
                opacity=1
            )
        )
    )

fig.update_layout(
    autosize=False,
    width=600,
    height=600,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    scene=dict(aspectmode='cube'),
    scene_camera = dict(
        up=dict(x=0,y=0,z=1),
        center=dict(x=0,y=0,z=0),
        eye=dict(x=0,y=-2,z=0.5)
    ),
)

# Show the plot
fig.show()

You can represent density as a surface. First slicing it:

In [118]:
slice = "itheta=0"
gasdens_plane,mesh = gasdens.meshslice(slice=slice)

And then plotting the surface:

In [124]:
fig = go.Figure()

fig.add_trace(
    go.Surface(
        x=mesh.x,
        y=mesh.y,
        z=np.log10(gasdens_plane),
        contours=dict(
            z=dict(
                show=True,
                usecolormap=True,
                highlightcolor="limegreen",
                project=dict(z=True)
            )
        ),
        colorscale='thermal',
        opacity=1,
        showscale=False
    )
)

fig.update_layout(
    autosize=False,
    width=600,
    height=600,
    scene=dict(
        aspectmode='cube',
        xaxis=dict(title='X'),
        yaxis=dict(title='Y'),
        zaxis=dict(title='Z'),
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.show()


---
*Powered by fargopy*. For more examples see [fargopy GitHub repo](https://github.com/seap-udea/fargopy/tree/main/examples). 

Jorge I. Zuluaga, Alejandro Murillo-González and Matías Montesinos © 2023-present
